# News Category Automation NLP

#Preprocessing

# Import Modules

In [56]:
import numpy as np
import pandas as pd
import textblob as TextBlob
import string
import nltk
import spacy
from nltk.corpus import stopwords
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "spacy"])
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 23.7 MB/s  0:00:00m0:00:010:01



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


0

# Load Data

In [57]:
file_path = '../data/processed/news_data_cleaned.csv'
data = pd.read_csv(file_path)
data.head()

,link,category,authors,date,description
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to the U.S. attorney's office in Los Angeles."
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,"23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23) ""Until you have a dog you don't understand what could be eaten."""
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,"The Funniest Tweets From Parents This Week (Sept. 17-23) ""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with a Carolina Reaper dipped in Tabasco sauce."""
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,Woman Who Called Cops On Black Bird-Watcher Loses Lawsuit Against Ex-Employer Amy Cooper accused investment firm Franklin Templeton of unfairly firing her and branding her a racist after video of the Central Park encounter went viral.


In [58]:
#examine data
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 189802 entries, 0 to 189801
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   link         189802 non-null  str  
 1   category     189802 non-null  str  
 2   authors      156860 non-null  str  
 3   date         189802 non-null  str  
 4   description  189802 non-null  str  
dtypes: str(5)
memory usage: 7.2 MB


In [59]:
#confirm missing description
data[data['description'] == '']

,link,category,authors,date,description


# Translate Spanish text

First, we'll translate Spanish text to ensure consistency in our data

In [60]:
pip install langdetect


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [61]:
pip install pandarallel


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [62]:
from pandarallel import pandarallel
# Initialize paralell processing
pandarallel.initialize()

# Check for Spanish text in Latino Voices
from langdetect import detect
data['language'] = data['description'].parallel_apply(lambda x: detect(x))
spanish = data[data['language'] == 'es']

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [63]:
# Check spanish descriptions
spanish

,link,category,authors,date,description,language
54255,https://www.huffingtonpost.com/entry/2016-nobel-peace-prize_us_57f7649ce4b068ecb5dd997d,News,NaN,2016-10-07,"2016 Nobel Peace Prize Awarded To Colombian President Juan Manuel Santos He is the 2nd Colombian-born Nobel Laureate, after writer Gabriel García Márquez.",es
75502,https://www.huffingtonpost.com/entry/barack-obama-serenades-hillarys-america-in-madame-president_us_56afb37be4b0b8d7c2301d3f,Entertainment,"Nadya Agrawal, Guest Writer",2016-02-01,"Barack Obama Serenades Hillary Clinton In Parody Endorsement Video Here's to you, Madame President.",es
80079,https://www.huffingtonpost.com/entry/gina-rodriguez-golden-globes-america-ferrera_us_5669e405e4b0f290e522866a,Entertainment,Julia Brucculieri,2015-12-10,"Gina Rodriguez Responds To Golden Globes' America Ferrera Mix-Up ""Who cares?""",es
95777,https://www.huffingtonpost.com/entry/vivan-los-amos-casa_b_7470398.html,Arts & Culture,"Hirania Luzardo, ContributorJournalist",2015-05-31,"¡Que vivan los amos de casa! Así como les sucede a muchas mamás, es natural que en algún momento el papá sienta la necesidad de salir de la casa a generar ingresos. Quedarse en casa de por sí es un gran compromiso de parte de los papás.",es
113395,https://www.huffingtonpost.com/entry/aaron-sanchez-do-cinco-de_b_5268578.html,Health,"American Food Roots, ContributorTelling the nation's food stories state by state",2014-05-05,Aaron Sanchez: Do Cinco de Mayo Like a Mexican Cinco de Mayo is more than an excuse to drink margaritas.,es


Looks like most of the spanish language is names of people and places. We'll translate this

In [64]:
pip install googletrans==4.0.0-rc1

  Using cached httpx-0.13.3-py3-none-any.whl.metadata (25 kB)
  Using cached httpcore-0.9.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached h11-0.9.0-py2.py3-none-any.whl.metadata (8.1 kB)
Using cached httpx-0.13.3-py3-none-any.whl (55 kB)
Using cached httpcore-0.9.1-py3-none-any.whl (42 kB)
Using cached h11-0.9.0-py2.py3-none-any.whl (53 kB)
  Attempting uninstall: h11
    Found existing installation: h11 0.16.0
    Uninstalling h11-0.16.0:
      Successfully uninstalled h11-0.16.0
  Attempting uninstall: httpcore
    Found existing installation: httpcore 1.0.9━━━━━━━━━━━━━━━━━━━ 1/3 [httpcore]
    Uninstalling httpcore-1.0.9:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [httpcore]
      Successfully uninstalled httpcore-1.0.9━━━━━━━━━━━━━━━━━ 1/3 [httpcore]
  Attempting uninstall: httpx0m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [httpcore]
    Found existing installation: httpx 0.28.1━━━━━━━━━━━━━━━━━ 1/3 [httpcore]
    Uninstalling httpx-0.28.1:m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [httpcore]
      Successfully 

In [65]:
from googletrans import Translator
translator = Translator()

In [66]:
# Translate spanish text
spanish['description'] = spanish['description'].parallel_apply(lambda x: translator.translate(x, dest='en').text)
spanish

,link,category,authors,date,description,language
54255,https://www.huffingtonpost.com/entry/2016-nobel-peace-prize_us_57f7649ce4b068ecb5dd997d,News,NaN,2016-10-07,"2016 Nobel Peace Prize Awarded To Colombian President Juan Manuel Santos He is the 2nd Colombian-born Nobel Laureate, after writer Gabriel García Márquez.",es
75502,https://www.huffingtonpost.com/entry/barack-obama-serenades-hillarys-america-in-madame-president_us_56afb37be4b0b8d7c2301d3f,Entertainment,"Nadya Agrawal, Guest Writer",2016-02-01,"Barack Obama Serenades Hillary Clinton In Parody Endorsement Video Here's to you, Madame President.",es
80079,https://www.huffingtonpost.com/entry/gina-rodriguez-golden-globes-america-ferrera_us_5669e405e4b0f290e522866a,Entertainment,Julia Brucculieri,2015-12-10,"Gina Rodriguez Responds To Golden Globes' America Ferrera Mix-Up ""Who cares?""",es
95777,https://www.huffingtonpost.com/entry/vivan-los-amos-casa_b_7470398.html,Arts & Culture,"Hirania Luzardo, ContributorJournalist",2015-05-31,"Long live the householders!Just as it happens to many mothers, it is natural that at some point the father feels the need to leave the house to generate income.Staying at home in itself is a great commitment on the part of parents.",es
113395,https://www.huffingtonpost.com/entry/aaron-sanchez-do-cinco-de_b_5268578.html,Health,"American Food Roots, ContributorTelling the nation's food stories state by state",2014-05-05,Aaron Sanchez: Do Cinco de Mayo Like a Mexican Cinco de Mayo is more than an excuse to drink margaritas.,es


Now the spanish text looks better so we can add it back into the original dataframe. I also notice there is some emojis here so we will make a note to take care of that later.

In [67]:
# Add translated text back to original df
data = pd.concat([data, spanish]).drop_duplicates()
data[data['language'] == 'es']

,link,category,authors,date,description,language
54255,https://www.huffingtonpost.com/entry/2016-nobel-peace-prize_us_57f7649ce4b068ecb5dd997d,News,NaN,2016-10-07,"2016 Nobel Peace Prize Awarded To Colombian President Juan Manuel Santos He is the 2nd Colombian-born Nobel Laureate, after writer Gabriel García Márquez.",es
75502,https://www.huffingtonpost.com/entry/barack-obama-serenades-hillarys-america-in-madame-president_us_56afb37be4b0b8d7c2301d3f,Entertainment,"Nadya Agrawal, Guest Writer",2016-02-01,"Barack Obama Serenades Hillary Clinton In Parody Endorsement Video Here's to you, Madame President.",es
80079,https://www.huffingtonpost.com/entry/gina-rodriguez-golden-globes-america-ferrera_us_5669e405e4b0f290e522866a,Entertainment,Julia Brucculieri,2015-12-10,"Gina Rodriguez Responds To Golden Globes' America Ferrera Mix-Up ""Who cares?""",es
95777,https://www.huffingtonpost.com/entry/vivan-los-amos-casa_b_7470398.html,Arts & Culture,"Hirania Luzardo, ContributorJournalist",2015-05-31,"¡Que vivan los amos de casa! Así como les sucede a muchas mamás, es natural que en algún momento el papá sienta la necesidad de salir de la casa a generar ingresos. Quedarse en casa de por sí es un gran compromiso de parte de los papás.",es
113395,https://www.huffingtonpost.com/entry/aaron-sanchez-do-cinco-de_b_5268578.html,Health,"American Food Roots, ContributorTelling the nation's food stories state by state",2014-05-05,Aaron Sanchez: Do Cinco de Mayo Like a Mexican Cinco de Mayo is more than an excuse to drink margaritas.,es
95777,https://www.huffingtonpost.com/entry/vivan-los-amos-casa_b_7470398.html,Arts & Culture,"Hirania Luzardo, ContributorJournalist",2015-05-31,"Long live the householders!Just as it happens to many mothers, it is natural that at some point the father feels the need to leave the house to generate income.Staying at home in itself is a great commitment on the part of parents.",es


# Lower Case Text

In [68]:
#lowercase description column
pd.set_option('display.max_colwidth', None)
data['description'] = data['description'].str.lower()
data.head()

,link,category,authors,date,description,language
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,over 4 million americans roll up sleeves for omicron-targeted covid boosters health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the u.s. ordered for the fall.,en
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,"american airlines flyer charged, banned for life after punching flight attendant on video he was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to the u.s. attorney's office in los angeles.",en
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,"23 of the funniest tweets about cats and dogs this week (sept. 17-23) ""until you have a dog you don't understand what could be eaten.""",en
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,"the funniest tweets from parents this week (sept. 17-23) ""accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like i was cleaning his teeth with a carolina reaper dipped in tabasco sauce.""",en
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,woman who called cops on black bird-watcher loses lawsuit against ex-employer amy cooper accused investment firm franklin templeton of unfairly firing her and branding her a racist after video of the central park encounter went viral.,en


This will ensure consistency in words and reduce vocab size to help our model.

# Check for URLs/HTML tags

In [69]:
import re

# Check for HTML tags
html_tags = data['description'].str.contains(r'<.*?>', regex=True)
html_tags[html_tags == True]

Series([], Name: description, dtype: bool)

In [70]:
# Check for URL
url = data['description'].str.contains(r'http\S+|www.\S+', regex=True)
url[url == True]

14384     True
16033     True
21359     True
23083     True
25327     True
          ... 
187426    True
187769    True
187982    True
188384    True
188967    True
Name: description, Length: 187, dtype: bool

In [71]:
# Remove URLs
data['description'] = data['description'].str.replace(r'http\S+|www.\S+', '', regex=True)
data['description'].str.contains(r'http\S+|www.\S+', regex=True).any()

np.False_

Now we have no html tags or urls in our text data

# Remove Punctuation

In [72]:
# Create punctuation variable from string
punc = string.punctuation
punc

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [73]:
# Remove punctuation from description
data['description'] = data['description'].str.translate(str.maketrans('', '', punc))
data.head()

,link,category,authors,date,description,language
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,over 4 million americans roll up sleeves for omicrontargeted covid boosters health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the us ordered for the fall,en
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,american airlines flyer charged banned for life after punching flight attendant on video he was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation according to the us attorneys office in los angeles,en
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,23 of the funniest tweets about cats and dogs this week sept 1723 until you have a dog you dont understand what could be eaten,en
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,the funniest tweets from parents this week sept 1723 accidentally put grownup toothpaste on my toddler’s toothbrush and he screamed like i was cleaning his teeth with a carolina reaper dipped in tabasco sauce,en
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,woman who called cops on black birdwatcher loses lawsuit against exemployer amy cooper accused investment firm franklin templeton of unfairly firing her and branding her a racist after video of the central park encounter went viral,en


In [74]:
# Double check for punctuation
punc_pattern = r'[{}]'.format(punc)
data['description'].str.contains(punc_pattern).any()

np.False_

We've confirmed we removed punctuation from our text data. This will remove noise from our text data and make it cleaner for the model

# Handle ChatWords & StopWords

ChatWords would be internet slang (EX: LOL, TMI, etc.). This shouldn't be very prevalent in our data set as it is a news site but these may come in to play in some categories such as comedy, weird news, and others.

In [75]:
# Common ChatWords found in github repository https://github.com/rishabhverma17/sms_slang_translator/blob/master/slang.txt
chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "B4N": "Bye For Now",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "ILU: I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "KISS": "Keep It Simple, Stupid",
    "LDR": "Long Distance Relationship",
    "LMAO": "Laugh My A.. Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA?": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laugher",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don't care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "TIME": "Tears in my eyes",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "BFF": "Best friends forever",
    "CSL": "Can't stop laughing"
}

In [76]:
# Convert chat words to text:
def chat_word_conversion(text):
    """Convert chat words to text"""
    new_text = []
    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)

In [77]:
# Convert chat words in description column
data['description'] = data['description'].apply(lambda x: chat_word_conversion(x))
data.head()

,link,category,authors,date,description,language
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,over 4 million americans roll up sleeves for omicrontargeted covid boosters health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the us ordered for the fall,en
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,american airlines flyer charged banned for life after punching flight attendant on video he was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation according to the us attorneys office in los angeles,en
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,23 of the funniest tweets about cats and dogs this week sept 1723 until you have a dog you dont understand what could be eaten,en
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,the funniest tweets from parents this week sept 1723 accidentally put grownup toothpaste on my toddler’s toothbrush and he screamed like i was cleaning his teeth with a carolina reaper dipped in tabasco sauce,en
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,woman who called cops on black birdwatcher loses lawsuit against exemployer amy cooper accused investment firm franklin templeton of unfairly firing her and branding her a racist after video of the central park encounter went viral,en


Now we will also handle StopWords like 'the', 'is', 'and', etc. These carry little meaning and removing them will reduce noise for our model

In [78]:
#download stopwords
nltk.download('stopwords')

[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1002)>


False

In [79]:
# Create variable for english stop words
stopword = stopwords.words('english')

In [80]:
# Remove stopwords
data['description'] = data['description'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stopword)]))
data.head()

,link,category,authors,date,description,language
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,4 million americans roll sleeves omicrontargeted covid boosters health experts said early predict whether demand would match 171 million doses new boosters us ordered fall,en
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,american airlines flyer charged banned life punching flight attendant video subdued passengers crew fled back aircraft confrontation according us attorneys office los angeles,en
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,23 funniest tweets cats dogs week sept 1723 dog dont understand could eaten,en
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,funniest tweets parents week sept 1723 accidentally put grownup toothpaste toddler’s toothbrush screamed like cleaning teeth carolina reaper dipped tabasco sauce,en
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,woman called cops black birdwatcher loses lawsuit exemployer amy cooper accused investment firm franklin templeton unfairly firing branding racist video central park encounter went viral,en


# Handle Emojis

In [81]:
pip install emoji


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [82]:
import emoji

# Remove emojis from description
data['description'] = data['description'].apply(lambda x: emoji.replace_emoji(x, replace=''))
data.iloc[36074]

link           https://www.huffingtonpost.com/entry/mike-pence-cinco-de-mayo-latino-trump_us_590c90a7e4b0104c734e6e8e
category                                                                                               Arts & Culture
authors                                                                                               Carolina Moreno
date                                                                                                       2017-05-05
description                                         mike pence uses cinco de mayo party claim latinos priority trump 
language                                                                                                           en
Name: 36074, dtype: str

#Tokenization

Tokenization will break down the text into managable words for processing and standardize the words

In [83]:
import re

# Use a simple regex tokenizer instead of NLTK punkt to avoid SSL download issues.
token_pattern = re.compile(r"\b\w[\w']*\b")
def simple_tokenize(text):
    if not isinstance(text, str):
        return []
    return token_pattern.findall(text)

In [84]:
# Tokenize the description column without NLTK punkt
data['description'] = data['description'].parallel_apply(simple_tokenize)

data.head()

,link,category,authors,date,description,language
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,"[4, million, americans, roll, sleeves, omicrontargeted, covid, boosters, health, experts, said, early, predict, whether, demand, would, match, 171, million, doses, new, boosters, us, ordered, fall]",en
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,"[american, airlines, flyer, charged, banned, life, punching, flight, attendant, video, subdued, passengers, crew, fled, back, aircraft, confrontation, according, us, attorneys, office, los, angeles]",en
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,"[23, funniest, tweets, cats, dogs, week, sept, 1723, dog, dont, understand, could, eaten]",en
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,"[funniest, tweets, parents, week, sept, 1723, accidentally, put, grownup, toothpaste, toddler, s, toothbrush, screamed, like, cleaning, teeth, carolina, reaper, dipped, tabasco, sauce]",en
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,"[woman, called, cops, black, birdwatcher, loses, lawsuit, exemployer, amy, cooper, accused, investment, firm, franklin, templeton, unfairly, firing, branding, racist, video, central, park, encounter, went, viral]",en


# Lemmatization

This will reduce words to their base form to enhance consistency. Lemmatization ensures words are transformed to their canonical form

In [85]:
# Download spaCy model for lemmatization
from spacy.cli import download
download("en_core_web_sm")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.2 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [88]:
# Load spaCy English model for lemmatization with only the needed components
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
print('Loaded spaCy model:', nlp.meta['name'])

Loaded spaCy model: core_web_sm


In [89]:
import os
from time import perf_counter

# Use spaCy pipe to lemmatize in batches and avoid per-row overhead
def spacy_lemmatize_pipe(texts, batch_size=1000, n_process=1):
    start = perf_counter()
    results = []
    for doc in nlp.pipe(texts, batch_size=batch_size, n_process=n_process):
        results.append([token.lemma_ for token in doc])
    elapsed = perf_counter() - start
    print(f"Processed {len(results)} documents in {elapsed:.1f}s (batch_size={batch_size}, n_process={n_process})")
    return results

texts = (" ".join(tokens) if isinstance(tokens, (list, tuple)) else str(tokens) for tokens in data['description'])
n_process = max(1, (os.cpu_count() or 2) - 1)
batch_size = 1000
data['description'] = spacy_lemmatize_pipe(texts, batch_size=batch_size, n_process=n_process)
data.head()

Processed 189803 documents in 683.7s (batch_size=1000, n_process=3)


,link,category,authors,date,description,language
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,"[4, million, americans, roll, sleeve, omicrontargete, covid, booster, health, expert, say, early, predict, whether, demand, would, match, 171, million, dos, new, booster, we, order, fall]",en
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,"[american, airlines, flyer, charge, ban, life, punch, flight, attendant, video, subdue, passenger, crew, flee, back, aircraft, confrontation, accord, us, attorney, office, los, angeles]",en
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,"[23, funniest, tweet, cat, dog, week, sept, 1723, dog, do, not, understand, could, eaten]",en
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,"[funniest, tweet, parent, week, sept, 1723, accidentally, put, grownup, toothpaste, toddler, s, toothbrush, scream, like, clean, teeth, carolina, reaper, dip, tabasco, sauce]",en
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,"[woman, call, cop, black, birdwatcher, lose, lawsuit, exemployer, amy, cooper, accuse, investment, firm, franklin, templeton, unfairly, fire, brand, racist, video, central, park, encounter, go, viral]",en


We'll also remove all words less thank two characters as these provide no little value to our data. We can see an 's' slipped through after an apostrophe. This code will also remove the apostrophes that weren't caught in our punctuation removal

In [90]:
# Remove all words < 2 char
data['description'] = data['description'].parallel_apply(lambda x: [word for word in x if len(word) > 2])
data.head()

,link,category,authors,date,description,language
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,News,"Carla K. Johnson, AP",2022-09-23,"[million, americans, roll, sleeve, omicrontargete, covid, booster, health, expert, say, early, predict, whether, demand, would, match, 171, million, dos, new, booster, order, fall]",en
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,News,Mary Papenfuss,2022-09-23,"[american, airlines, flyer, charge, ban, life, punch, flight, attendant, video, subdue, passenger, crew, flee, back, aircraft, confrontation, accord, attorney, office, los, angeles]",en
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,Entertainment,Elyse Wanshel,2022-09-23,"[funniest, tweet, cat, dog, week, sept, 1723, dog, not, understand, could, eaten]",en
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,Lifestyle,Caroline Bologna,2022-09-23,"[funniest, tweet, parent, week, sept, 1723, accidentally, put, grownup, toothpaste, toddler, toothbrush, scream, like, clean, teeth, carolina, reaper, dip, tabasco, sauce]",en
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,News,Nina Golgowski,2022-09-22,"[woman, call, cop, black, birdwatcher, lose, lawsuit, exemployer, amy, cooper, accuse, investment, firm, franklin, templeton, unfairly, fire, brand, racist, video, central, park, encounter, viral]",en


# Export data

In [ ]:
data.to_csv('../data/processed/news_data_processed.csv', index=False)